<a href="https://colab.research.google.com/github/Deepa-12345678/CHATBOT-FOR-GENERAL-HEALTH-QUERIES/blob/main/snlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pypdf chromadb sentence-transformers langchain-text-splitters google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

1. pypdf- Used for reading and extracting text from PDF files.

2. chromadb- Vector database used to store and search embeddings.

3. sentence-transformers- Converts text into embeddings (numerical vectors).

4. langchain-text-splitters- Splits large documents into smaller chunks.

5. google-genai - Official Google Gemini API SDK.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving hypertension.pdf to hypertension.pdf
Saving asthma.pdf to asthma.pdf
Saving diabetes.pdf to diabetes.pdf


Step 2: Upload Medical PDFs

In [ ]:
import os
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

# Read PDFs
all_text = ""

pdf_files = [f for f in os.listdir() if f.endswith(".pdf")]

for pdf in pdf_files:
    reader = PdfReader(pdf)

    for page in reader.pages:
        text = page.extract_text()

        if text:
            all_text += text + "\n"

print("Text extracted")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = splitter.split_text(all_text)

print("Chunks:", len(chunks))

# Embedding model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)

print("Embeddings created")

# Persistent Chroma DB
client_db = chromadb.PersistentClient(
    path="./medical_db"
)

collection = client_db.get_or_create_collection(
    name="medical_knowledge_base"
)

# Clear old documents if needed
try:
    existing = collection.count()

    if existing > 0:
        print("Collection already exists")
    else:

        for i, chunk in enumerate(chunks):

            collection.add(
                ids=[str(i)],
                documents=[chunk],
                embeddings=[embeddings[i].tolist()]
            )

except:

    for i, chunk in enumerate(chunks):

        collection.add(
            ids=[str(i)],
            documents=[chunk],
            embeddings=[embeddings[i].tolist()]
        )

print("Database ready")
print("Documents:", collection.count())

Text extracted
Chunks: 179


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created
Database ready
Documents: 179


In [ ]:
from google import genai
from getpass import getpass

GEMINI_API_KEY = getpass("Enter Gemini API Key: ")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini connected")

Enter Gemini API Key: ··········
Gemini connected


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

client_db = chromadb.PersistentClient(
    path="./medical_db"
)

collection = client_db.get_collection(
    "medical_knowledge_base"
)

print("Database loaded")
print("Documents:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Database loaded
Documents: 179


In [ ]:
# Save a sample of the knowledge base

sample_docs = collection.get(
    limit=min(20, collection.count()),
    include=["documents"]
)

knowledge_summary = "\n\n".join(
    sample_docs["documents"]
)

print("Knowledge summary created.")

Knowledge summary created.


In [ ]:
import time

def healthcare_chatbot(query):

    # Retrieve relevant chunks
    query_embedding = embedding_model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=5
    )

    context = "\n\n".join(results["documents"][0])

    prompt = f"""
You are HealthAssist, a friendly and knowledgeable healthcare AI assistant.

Use ONLY the medical context provided below.

Instructions:
- Answer in a natural, conversational way.
- Explain medical terms in simple language when possible.
- Use bullet points for symptoms, causes, and treatments.
- If the answer is not available in the context, politely say:
  "I don't have enough information about that topic in my current medical knowledge base."
- End with:
  "Note: This information is for educational purposes and is not a substitute for professional medical advice."

Medical Context:
{context}

Patient Question:
{query}

Assistant Response:
"""

    # Retry logic for Gemini 503 errors
    for attempt in range(5):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )

            return response.text

        except Exception as e:
            print(f"Attempt {attempt+1} failed:", e)

            if attempt < 4:
                time.sleep(5)

    return "Gemini service is temporarily unavailable. Please try again later."

In [ ]:
while True:

    query = input("Ask Question: ")

    if query.lower() == "exit":
        break

    answer = healthcare_chatbot(query)

    print("\nAnswer:\n")
    print("\n" + "="*60)
    print("🩺 HEALTHASSIST")
    print("="*60)
    print(answer)
    print("="*60)

Ask Question: what is hypertension

Answer:


🩺 HEALTHASSIST
Hello there! I can certainly tell you about hypertension based on my information.

Hypertension, also commonly known as high blood pressure, means your blood pressure is persistently higher than normal.

Here's how it's generally defined and diagnosed:

*   **Definition:** Blood pressure is considered elevated when it's consistently above 140 mmHg (millimeters of mercury) for the systolic reading and/or above 90 mmHg for the diastolic reading.
    *   **Systolic pressure:** This is the top number in a blood pressure reading and measures the pressure in your arteries when your heart beats.
    *   **Diastolic pressure:** This is the bottom number and measures the pressure in your arteries when your heart rests between beats.
*   **Diagnosis:**
    *   Diagnosis is usually based on high blood pressure measurements taken under standard conditions on at least three different occasions.
    *   Your blood pressure should always be